In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

# EDA

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [4]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Evaluation Metric

In [7]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

# Cross-Val Split

In [8]:
train_df, val_df = train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=4524)

# Model 1(Scratch): TfidfVectorizer

In [9]:
train_data_lr = []

for _, row in train_df.iterrows():
    prompt = str(row['prompt'])
    correct_answer = row['answer'] 
    
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_text = str(row[option])
        is_correct = 1 if option == correct_answer else 0
        text_combined = prompt + " " + option_text
        
        train_data_lr.append({
            'text': text_combined,
            'label': is_correct
        })

df_train_lr = pd.DataFrame(train_data_lr)
df_train_lr.head()

,text,label
0,Which of the following is correct? What is the...,0
1,Which of the following is correct? What is the...,0
2,Which of the following is correct? What is the...,1
3,Which of the following is correct? What is the...,0
4,Which of the following is correct? What is the...,0


In [10]:
all_text = []

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_text.extend(train[col].fillna('').astype(str))
    all_text.extend(test[col].fillna('').astype(str))

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

vectorizer.fit(all_text)

TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

In [11]:
X_train_tfidf = vectorizer.transform(df_train_lr['text'])
y_train = df_train_lr['label']

lr_model = LogisticRegression(class_weight='balanced',C=0.1, random_state=4524)
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(C=0.1, class_weight='balanced', random_state=4524)

In [12]:
def predict_top3_lr(df, vectorizer, model):
    predictions = []
    
    for _, row in df.iterrows():
        prompt = str(row['prompt'])
        scores = []
        
        for option in ['A', 'B', 'C', 'D', 'E']:
            text_combined = prompt + " " + str(row[option])
            vec = vectorizer.transform([text_combined])
            
            prob = model.predict_proba(vec)[0][1]
            scores.append((option, prob))
            
        scores.sort(key=lambda x: x[1], reverse=True)
        predictions.append([x[0] for x in scores[:3]])
        
    return predictions

In [13]:
val_preds = predict_top3_lr(val_df, vectorizer, lr_model)
val_actuals = val_df['answer'].tolist()

val_score = map3(val_actuals, val_preds)
print("Validation MAP@3:", val_score)

Validation MAP@3: 0.9554166666666667


# Submission Cell

In [14]:
test_predictions = predict_top3_lr(test, vectorizer, lr_model)

test_predictions = [" ".join(pred) for pred in test_predictions]

In [15]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A E C
1,2,B A C
2,3,B E D
3,4,E C A
4,5,C A B
